El paquete de _Python_ [scikit-learn](http://scikit-learn.org) (_sklearn_ en lo que sigue) proporciona un marco de trabajo para el aprendizaje automático.

# Aprendizaje supervisado

In [ ]:
Intesidad wifi. Compuesto por: Mohammed El Arfaoui Chiboub y Antonio Eugenio García Parejo

Para reducir la cantidad de valores posibles, la intensidad de la señal (en dBm) se ha discretizado dividiéndola entre 10 y redondeando (por ejemplo, `-64` pasa a ser `-6` y `-68` pasa a ser `-7`). Cuanto más cerca de 0, más fuerte es la señal.

* _WS1_: intensidad de la señal del router 1. Posibles valores: `-1`, `-2`, `-3`, `-4`, `-5`, `-6`, `-7`.
* _WS2_: intensidad de la señal del router 2. Posibles valores: `-5`, `-6`, `-7`.
* _WS3_: intensidad de la señal del router 3. Posibles valores: `-4`, `-5`, `-6`, `-7`.
* _WS4_: intensidad de la señal del router 4. Posibles valores: `-1`, `-2`, `-3`, `-4`, `-5`, `-6`, `-7`, `-8`.
* _WS5_: intensidad de la señal del router 5. Posibles valores: `-4`, `-5`, `-6`, `-7`, `-8`, `-9`.
* _WS6_: intensidad de la señal del router 6. Posibles valores: `-6`, `-7`, `-8`, `-9`, `-10`.
* _WS7_: intensidad de la señal del router 7. Posibles valores: `-6`, `-7`, `-8`, `-9`, `-10`.

La habitación en la que se encontraba el dispositivo se indica mediante el atributo _Room_, que puede tomar los valores `1`, `2`, `3` o `4`.

El objetivo es predecir, a partir de la intensidad de la señal recibida de los 7 routers, en cuál de las 4 habitaciones se encuentra el dispositivo. Se trata, por tanto, de un problema de clasificación.

Para leer los datos desde el fichero `wifi_localization_discretizado.csv` que se proporciona se pueden evaluar las siguientes expresiones ([_Pandas_](http://pandas.pydata.org/) y [_NumPy_](http://www.numpy.org/) son paquetes de _Python_ para análisis de datos y cálculo científico, respectivamente):

In [1]:
# Instalar paquetes pandas y numpy, pip install pandas numpy
import pandas
import numpy

wifi = pandas.read_csv('wifi_localization_discretizado.csv', header=None,
                       names=['WS1', 'WS2', 'WS3', 'WS4', 'WS5', 'WS6', 'WS7', 'Room'])

In [2]:
# Número de filas y columnas
print(wifi.shape)

(2000, 8)


In [3]:
# 10 primeras filas
wifi.head(10)

,WS1,WS2,WS3,WS4,WS5,WS6,WS7,Room
0,-6,-6,-6,-7,-7,-8,-8,1
1,-7,-6,-6,-7,-7,-9,-9,1
2,-6,-6,-6,-7,-8,-9,-8,1
3,-6,-6,-7,-6,-8,-9,-8,1
4,-6,-7,-6,-6,-8,-8,-9,1
5,-6,-6,-6,-7,-8,-9,-8,1
6,-7,-6,-7,-7,-7,-9,-8,1
7,-6,-6,-6,-7,-7,-9,-8,1
8,-7,-6,-6,-6,-8,-9,-8,1
9,-6,-6,-7,-7,-8,-9,-9,1


_sklearn_ no puede trabajar directamente con el conjunto de datos anterior, ya que asume que los valores de las variables discretas están codificadas con números enteros. Para transformar los datos a un formato adecuado ofrece diversas operaciones de preprocesamiento, entre las que se encuentran `OrdinalEncoder`, para codificar los atributos, y `LabelEncoder`, para codificar la variable objetivo.

In [4]:
# Instalar paquete scikit-learn, pip install scikit-learn
from sklearn import preprocessing

atributos = wifi.loc[:, 'WS1':'WS7']  # selección de las columnas de atributos
objetivo = wifi['Room']  # selección de la columna objetivo

# Para realizar una codificación de los datos, se crea una instancia del tipo de
# codificación pretendida y se ajusta a los datos disponibles mediante el método
# fit. Los métodos transform e inverse_transform permiten entonces codificar y
# descodificar, respectivamente, los datos.

In [5]:
# El codificador adecuado para los atributos es OrdinalEncoder, ya que permite
# trabajar con el array completo de valores de los atributos.
codificador_atributos = preprocessing.OrdinalEncoder()
codificador_atributos.fit(atributos)

# Categorías detectadas por el codificador para cada atributo
print(codificador_atributos.categories_)

[array([-7, -6, -5, -4, -3, -2, -1]), array([-7, -6, -5]), array([-7, -6, -5, -4]), array([-8, -7, -6, -5, -4, -3, -2, -1]), array([-9, -8, -7, -6, -5, -4]), array([-10,  -9,  -8,  -7,  -6]), array([-10,  -9,  -8,  -7,  -6])]


In [6]:
# Codificación de los atributos una vez ajustado el codificador
atributos_codificados = codificador_atributos.transform(atributos)
print(atributos_codificados)

[[1. 1. 1. ... 2. 2. 2.]
 [0. 1. 1. ... 2. 1. 1.]
 [1. 1. 1. ... 1. 1. 2.]
 ...
 [1. 1. 2. ... 4. 1. 1.]
 [1. 1. 2. ... 5. 1. 1.]
 [1. 2. 2. ... 4. 1. 1.]]


In [7]:
# Las respuestas predichas por los modelos estarán codificadas, pero el codificador
# también permite descodificarlas
print(codificador_atributos.inverse_transform([[1., 1., 1., 1., 2., 2., 2.],
                                               [1., 2., 2., 2., 4., 1., 1.]]))

[[-6 -6 -6 -7 -7 -8 -8]
 [-6 -5 -5 -6 -5 -9 -9]]


In [8]:
# El codificador adecuado para la variable objetivo es LabelEncoder, que trabaja
# con la lista de sus valores, en lugar de con un array.
codificador_objetivo = preprocessing.LabelEncoder()

# El método fit_transform ajusta la codificación y la aplica a los datos a continuación
objetivo_codificado = codificador_objetivo.fit_transform(objetivo)

In [9]:
# Clases detectadas por el codificador para la variable objetivo
print(codificador_objetivo.classes_)

[1 2 3 4]


In [10]:
print(objetivo_codificado)

[0 0 0 ... 3 3 3]


In [11]:
print(codificador_objetivo.inverse_transform([2, 1, 3]))

[3 2 4]


Una vez preparadas las variables, es necesario separar el conjunto de datos en dos: un conjunto de entrenamiento, que se usará para construir los distintos modelos; y un conjunto de prueba, que se usará para comparar los distintos modelos.

En este caso, la distribución de ejemplos en las distintas habitaciones es uniforme: hay 500 mediciones (un 25 % del total) en cada una de las habitaciones `1`, `2`, `3` y `4`.

In [12]:
# Cantidad total de ejemplos
print(wifi.shape[0])

2000


In [13]:
# Frecuencia total de cada habitación
print(pandas.Series(objetivo).value_counts(normalize=True))

Room
1    0.25
2    0.25
3    0.25
4    0.25
Name: proportion, dtype: float64


Aunque las clases están equilibradas, es conveniente que la separación de los ejemplos se realice de manera estratificada, es decir, manteniendo la proporción anterior (25 % por habitación) tanto en el conjunto de entrenamiento como en el de prueba.

Para dividir un conjunto de datos en un subconjunto de entrenamiento y otro de prueba, _sklearn_ proporciona la función `train_test_split`.

In [14]:
from sklearn import model_selection

(atributos_entrenamiento, atributos_prueba,
 objetivo_entrenamiento, objetivo_prueba) = model_selection.train_test_split(
        # Conjuntos de datos a dividir, usando los mismos índices para ambos
        atributos_codificados, objetivo_codificado,
        # Valor de la semilla aleatoria, para que el muestreo sea reproducible,
        # a pesar de ser aleatorio
        random_state=12345,
        # Tamaño del conjunto de prueba
        test_size=.33,
        # Estratificamos respecto a la distribución de valores en la variable objetivo
        stratify=objetivo_codificado)

In [15]:
# Comprobamos que el conjunto de prueba contiene el 33 % de los datos, en la misma proporción
# con respecto a la variable objetivo
print("Cantidad de ejemplos de pruebas requeridos:", 2000 * .33)
print("Filas del array de atributos de prueba:", atributos_prueba.shape[0])
print("Longitud del vector de objetivos de prueba:", len(objetivo_prueba))
print("Proporción de clases en el vector de objetivos de prueba:")
print(pandas.Series(
        codificador_objetivo.inverse_transform(objetivo_prueba)
      ).value_counts(normalize=True))

Cantidad de ejemplos de pruebas requeridos: 660.0
Filas del array de atributos de prueba: 660
Longitud del vector de objetivos de prueba: 660
Proporción de clases en el vector de objetivos de prueba:
4    0.25
1    0.25
2    0.25
3    0.25
Name: proportion, dtype: float64


In [16]:
# Comprobamos que el conjunto de entrenamiento contiene el resto de los datos, en la misma
# proporción con respecto a la variable objetivo
print("Cantidad de ejemplos de entrenamiento requeridos:", 2000 * .67)
print("Filas del array de atributos de entrenamiento:", atributos_entrenamiento.shape[0])
print("Longitud del vector de objetivos de entrenamiento:", len(objetivo_entrenamiento))
print("Proporción de clases en el vector de objetivos de entrenamiento:")
print(pandas.Series(
        codificador_objetivo.inverse_transform(objetivo_entrenamiento)
      ).value_counts(normalize=True))

Cantidad de ejemplos de entrenamiento requeridos: 1340.0
Filas del array de atributos de entrenamiento: 1340
Longitud del vector de objetivos de entrenamiento: 1340
Proporción de clases en el vector de objetivos de entrenamiento:
1    0.25
3    0.25
4    0.25
2    0.25
Name: proportion, dtype: float64


Para realizar aprendizaje supervisado en _sklearn_ basta crear una instancia de la clase de objetos que implemente el modelo que se quiera utilizar (árboles de decisión, _naive_ Bayes, _kNN_, etc.).

Cada una de estas instancias dispondrá de los siguientes métodos:
* El método `fit` permite entrenar el modelo, dados __por separado__ el conjunto de ejemplos de entrenamiento y la clase de cada uno de estos ejemplos.
* El método `predict` permite clasificar un nuevo ejemplo una vez entrenado el modelo.
* El método `score` calcula el rendimiento del modelo, dados __por separado__ el conjunto de ejemplos de prueba y la clase de cada uno de estos ejemplos.

### Árboles de decisión

_sklearn_ implementa los árboles de decisión clasificadores como instancias de la clase `DecisionTreeClassifier`.

Desafortunadamente, son árboles de decisión binarios construidos asumiendo atributos continuos y mediante un algoritmo distinto a _ID3_, que no está implementado.

En http://scikit-learn.org/stable/modules/tree.html se puede encontrar información acerca de los árboles de decisión implementados en _sklearn_.

In [17]:
from sklearn import preprocessing
from sklearn import model_selection
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

# En scikit-learn, ID3 se especifica mediante criterion='entropy'.
# Nota: La implementación nativa de sklearn soporta divisiones numéricas/ordinales.
modelo_id3 = DecisionTreeClassifier(
    criterion='entropy',  # Usa Ganancia de Información (Entropía) como en ID3
    random_state=12345
)

# Entrenamiento del modelo
modelo_id3.fit(atributos_entrenamiento, objetivo_entrenamiento)

# 7. Evaluación del modelo
predicciones = modelo_id3.predict(atributos_prueba)

# Métricas de precisión general
precision_general = accuracy_score(objetivo_prueba, predicciones)
print(f"Exactitud (Accuracy) del modelo ID3: {precision_general * 100:.2f}%\n")

# Reporte detallado por clase (Precision, Recall, F1-Score)
etiquetas_clases = [f"Habitación {clase}" for clase in codificador_objetivo.classes_]
print("=== Reporte de Clasificación ===")
print(classification_report(
    objetivo_prueba, 
    predicciones, 
    target_names=etiquetas_clases
))

# 8. Visualización en texto de la estructura del árbol
print("\n=== Reglas del Árbol de Decisión ===")
reglas_arbol = export_text(modelo_id3, feature_names=list(atributos.columns))
print(reglas_arbol[:1000])  # Muestra las primeras reglas generadas

Exactitud (Accuracy) del modelo ID3: 97.27%

=== Reporte de Clasificación ===
              precision    recall  f1-score   support

Habitación 1       0.98      0.99      0.98       165
Habitación 2       1.00      0.96      0.98       165
Habitación 3       0.95      0.97      0.96       165
Habitación 4       0.96      0.97      0.97       165

    accuracy                           0.97       660
   macro avg       0.97      0.97      0.97       660
weighted avg       0.97      0.97      0.97       660


=== Reglas del Árbol de Decisión ===
|--- WS1 <= 1.50
|   |--- WS5 <= 3.50
|   |   |--- WS5 <= 2.50
|   |   |   |--- WS4 <= 2.50
|   |   |   |   |--- class: 0
|   |   |   |--- WS4 >  2.50
|   |   |   |   |--- class: 2
|   |   |--- WS5 >  2.50
|   |   |   |--- WS7 <= 1.50
|   |   |   |   |--- WS6 <= 1.50
|   |   |   |   |   |--- WS3 <= 1.50
|   |   |   |   |   |   |--- class: 3
|   |   |   |   |   |--- WS3 >  1.50
|   |   |   |   |   |   |--- WS4 <= 1.50
|   |   |   |   |   |   |   

In [18]:
# =============================================================================
# 9. Predicción con datos nuevos
# =============================================================================

# Definimos varias mediciones nuevas de señal WiFi (discretizadas)
nuevas_mediciones = pandas.DataFrame([
    {'WS1': -6, 'WS2': -6, 'WS3': -6, 'WS4': -7, 'WS5': -7, 'WS6': -8, 'WS7': -8},
    {'WS1': -4, 'WS2': -6, 'WS3': -4, 'WS4': -4, 'WS5': -6, 'WS6': -8, 'WS7': -9},
    {'WS1': -5, 'WS2': -5, 'WS3': -5, 'WS4': -5, 'WS5': -6, 'WS6': -8, 'WS7': -8},
    {'WS1': -6, 'WS2': -6, 'WS3': -6, 'WS4': -6, 'WS5': -5, 'WS6': -9, 'WS7': -9}
])

print("--- Datos nuevos a evaluar ---")
print(nuevas_mediciones)

# Codificamos los atributos con el mismo codificador usado en el entrenamiento
nuevos_atributos_codificados = codificador_atributos.transform(nuevas_mediciones)

# Realizamos la predicción numérica con el modelo ID3
predicciones_numericas = modelo_id3.predict(nuevos_atributos_codificados)

# Convertimos la predicción numérica (0-3) de vuelta a su habitación original (1-4)
predicciones_etiquetas = codificador_objetivo.inverse_transform(predicciones_numericas)

# Mostramos los resultados en un DataFrame claro
resultados = nuevas_mediciones.copy()
resultados['prediccion_habitacion'] = predicciones_etiquetas

print("\n--- Resultados de la predicción ---")
print(resultados)

--- Datos nuevos a evaluar ---
   WS1  WS2  WS3  WS4  WS5  WS6  WS7
0   -6   -6   -6   -7   -7   -8   -8
1   -4   -6   -4   -4   -6   -8   -9
2   -5   -5   -5   -5   -6   -8   -8
3   -6   -6   -6   -6   -5   -9   -9

--- Resultados de la predicción ---
   WS1  WS2  WS3  WS4  WS5  WS6  WS7  prediccion_habitacion
0   -6   -6   -6   -7   -7   -8   -8                      1
1   -4   -6   -4   -4   -6   -8   -9                      2
2   -5   -5   -5   -5   -6   -8   -8                      3
3   -6   -6   -6   -6   -5   -9   -9                      4


_sklearn_ implementa _naive_ Bayes para atributos categóricos mediante instancias de la clase `CategoricalNB`.

In [19]:
from sklearn import naive_bayes

clasif_NB = naive_bayes.CategoricalNB(alpha=1.0)  # alpha es el parámetro de suavizado
clasif_NB.fit(atributos_entrenamiento, objetivo_entrenamiento)

,"alpha alpha: float, default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
,"min_categories min_categories: int or array-like of shape (n_features,), default=NoneMinimum number of categories per feature.- integer: Sets the minimum number of categories per feature to `n_categories` for each features.- array-like: shape (n_features,) where `n_categories[i]` holds the minimum number of categories for the ith column of the input.- None (default): Determines the number of categories automatically from the training data... versionadded:: 0.24",None
Name,Type,Value
"category_count_ category_count_: list of arrays of shape (n_features,)Holds arrays of shape (n_classes, n_categories of respective feature)for each feature. Each array provides the number of samplesencountered for each class and category of the specific feature.",list,"[array([[ 87.,... 0., 0.]]), array([[ 3.,... 199., 134.]]), array([[ 49.,... 239., 20.]]), array([[ 4.,... 0., 0.]]), ...]"
"class_count_ class_count_: ndarray of shape (n_classes,)Number of samples encountered for each class during fitting. Thisvalue is weighted by the sample weight when provided.","ndarray[float64](4,)","[335.,335.,335.,335.]"
"class_log_prior_ class_log_prior_: ndarray of shape (n_classes,)Smoothed empirical log probability for each class.","ndarray[float64](4,)","[-1.39,-1.39,-1.39,-1.39]"
"classes_ classes_: ndarray of shape (n_classes,)Class labels known to the classifier","ndarray[int64](4,)","[0,1,2,3]"
"feature_log_prob_ feature_log_prob_: list of arrays of shape (n_features,)Holds arrays of shape (n_classes, n_categories of respective feature)for each feature. Each array provides the empirical log probabilityof categories given the respective feature and class, ``P(x_i|y)``.",list,"[array([[-1.35...-5.83481074]]), array([[-4.43...-0.91777112]]), array([[-1.91...-2.78147767]]), array([[-4.22...-5.83773045]]), ...]"


Las siguientes expresiones muestran las cuentas realizadas y (los logaritmos de) las probabilidades aprendidas por el modelo.

In [20]:
for clase, cantidad_ejemplos_clase, log_probabilidad_clase in zip(
    codificador_objetivo.classes_, clasif_NB.class_count_, clasif_NB.class_log_prior_):
        print("Cantidad de ejemplos para la clase", clase, ":", cantidad_ejemplos_clase)
        print("Logaritmo de la probabilidad aprendida para la clase", clase, ":", log_probabilidad_clase)

Cantidad de ejemplos para la clase 1 : 335.0
Logaritmo de la probabilidad aprendida para la clase 1 : -1.3862943611198908
Cantidad de ejemplos para la clase 2 : 335.0
Logaritmo de la probabilidad aprendida para la clase 2 : -1.3862943611198908
Cantidad de ejemplos para la clase 3 : 335.0
Logaritmo de la probabilidad aprendida para la clase 3 : -1.3862943611198908
Cantidad de ejemplos para la clase 4 : 335.0
Logaritmo de la probabilidad aprendida para la clase 4 : -1.3862943611198908


In [21]:
for i, atributo in enumerate(cars.columns[:-1]):
    print("Cantidad de ejemplos para el atributo", atributo, ":")
    print(pandas.DataFrame(clasif_NB.category_count_[i],
                          columns=codificador_atributos.categories_[i],
                          index=codificador_objetivo.classes_))
    print("Logaritmos de las probabilidades aprendidas para el atributo", atributo, ":")
    print(pandas.DataFrame(clasif_NB.feature_log_prob_[i],
                          columns=codificador_atributos.categories_[i],
                          index=codificador_objetivo.classes_))
    print()

NameError: name 'cars' is not defined

El método `predict` devuelve la clase predicha por el modelo para un nuevo ejemplo y el método `score` la exactitud (_accuracy_) media sobre un conjunto de datos de prueba.

In [22]:
nuevos_ejemplos = pandas.DataFrame([[-6, -6, -6, -7, -7, -8, -8],
                                    [-4, -6, -4, -4, -6, -8, -9]],
                                   columns=atributos.columns)
codificador_objetivo.inverse_transform(clasif_NB.predict(codificador_atributos.transform(nuevos_ejemplos)))

array([1, 2])

In [23]:
# Calculamos la fracción de clases correctamente predichas para el conjunto de datos de prueba
clasif_NB.score(atributos_prueba, objetivo_prueba)

0.9712121212121212